In [1]:
pip install numpy matplotlib scikit-learn tensorflow seaborn


Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load preprocessed data
x_train = np.load('./data_aligned/x_train.npy')
x_test = np.load('./data_aligned/x_test.npy')
y_train = np.load('./data_aligned/y_train.npy')
y_test = np.load('./data_aligned/y_test.npy')

# Check shapes
print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

# CNN expects channel last: (samples, height, width, channels)
x_train = x_train.reshape((-1, 19, 500, 1))
x_test = x_test.reshape((-1, 19, 500, 1))

# One-hot encode labels
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=4)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=4)

# Build CNN model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(19, 500, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.BatchNormalization(),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.BatchNormalization(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')  # 4 classes
])

# Compile model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train model
history = model.fit(x_train, y_train_cat, epochs=10, batch_size=64,
                    validation_data=(x_test, y_test_cat))

# Evaluate model
loss, accuracy = model.evaluate(x_test, y_test_cat)
print(f"Test Accuracy: {accuracy:.4f}")

# Classification report
y_pred = model.predict(x_test)
y_pred_labels = np.argmax(y_pred, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_labels, target_names=['Normal', 'CPS', 'ELEC', 'NOC']))

# Confusion Matrix
conf_mat = confusion_matrix(y_test, y_pred_labels)
plt.figure(figsize=(8,6))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'CPS', 'ELEC', 'NOC'],
            yticklabels=['Normal', 'CPS', 'ELEC', 'NOC'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: './data_aligned/x_train.npy'